<h1>03 - Looking Inside Transformer LLMs</h1>
<i>An extensive look into the transformer architecture of generative LLMs</i>


In [1]:
# %%capture
# !pip install transformers>=4.41.2 accelerate>=0.31.0

# Loading the LLM

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

In [3]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
device = "mps" if torch.backends.mps.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map=device,
    torch_dtype="auto",
    trust_remote_code=False,
)

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=100,
    do_sample=False,
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# making Ram free from misusing via garbage collector

In [4]:
# import gc
# del model
# del generator
# del tokenizer
# gc.collect()
# torch.mps.empty_cache()
# pipeline

# The Inputs and Outputs of a Trained Transformer LLM


In [5]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

You are not running the flash-attention implementation, expect numerical differences.


 Mention the steps you're taking to prevent it in the future.

Dear Sarah,

I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden. It was a tragic mishap that I never intended, and I am truly sorry for any distress it may have caused you.

The incident happened when I was attempting to help you with your gardening project. I


In [6]:
type(output)

list

In [7]:
print(output)

[{'generated_text': " Mention the steps you're taking to prevent it in the future.\n\nDear Sarah,\n\nI hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden. It was a tragic mishap that I never intended, and I am truly sorry for any distress it may have caused you.\n\nThe incident happened when I was attempting to help you with your gardening project. I"}]


In [8]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=3206

# Choosing a single token from the probability distribution (sampling / decoding)

In [9]:
prompt = "The capital of France is"
device='mps'
# Tokenize the input prompt
encoded=tokenizer(prompt,return_tensors='pt')

# Tokenize the input prompt
input_ids = encoded['input_ids'].to(device)

# Get the output of the model before the lm_head
model_output = model.model(input_ids)

# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])

In [20]:
print(encoded.input_ids)

tensor([[14350,   263,  1407,  1472,  4876, 27746,  5281,   304, 19235,   363,
           278, 25305,   293, 16423,   292,   286,   728,   481, 29889, 12027,
          7420,   920,   372,  9559, 29889]])


In [22]:
for i in encoded.input_ids:
    print(tokenizer.decode(i))

Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.


In [10]:
type(model)

transformers.models.phi3.modeling_phi3.Phi3ForCausalLM

In [11]:
model_output

BaseModelOutputWithPast(last_hidden_state=tensor([[[-0.3027,  1.1797,  0.3047,  ..., -0.3027,  0.6875,  0.1260],
         [-0.1299,  0.3105,  0.3652,  ...,  0.5547, -0.1328, -0.5938],
         [-0.5820,  1.0781,  1.5469,  ..., -0.4219,  0.2676,  0.4121],
         [-0.3926,  0.6562,  0.2471,  ..., -0.0635,  0.8008, -0.6836],
         [-0.6641,  0.7109,  0.5391,  ...,  0.2637,  0.1943, -0.2812]]],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<MulBackward0>), past_key_values=((tensor([[[[-2.9688e-01,  1.3770e-01, -7.8125e-02,  ..., -9.1797e-02,
           -2.6123e-02,  5.8350e-02],
          [ 8.6914e-02,  9.7656e-02, -1.3672e-01,  ..., -2.3926e-02,
           -1.0840e-01, -2.7100e-02],
          [ 4.0820e-01,  4.1504e-02, -1.9141e-01,  ..., -1.5137e-01,
           -2.1094e-01,  8.7891e-02],
          [ 2.5977e-01, -3.2227e-02, -3.8330e-02,  ..., -1.6602e-02,
           -1.2268e-02,  6.3782e-03],
          [ 9.8633e-02, -9.8633e-02, -7.1289e-02,  ..., -1.9336e-01,
            2.30

In [12]:
lm_head_output

tensor([[[25.0000, 25.1250, 23.0000,  ..., 19.1250, 19.1250, 19.1250],
         [31.0000, 31.5000, 26.0000,  ..., 25.8750, 25.8750, 25.8750],
         [31.5000, 28.8750, 31.0000,  ..., 26.2500, 26.2500, 26.2500],
         [33.0000, 31.8750, 36.0000,  ..., 27.8750, 27.8750, 27.8750],
         [27.8750, 29.5000, 28.2500,  ..., 20.5000, 20.5000, 20.5000]]],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<LinearBackward0>)

In [28]:
token_id = lm_head_output[0,-1].argmax()
token_id

tensor(3681, device='mps:0')

In [13]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

In [14]:
model_output[0].shape

torch.Size([1, 5, 3072])

In [15]:
lm_head_output.shape

torch.Size([1, 5, 32064])

In [23]:
type(lm_head_output)

torch.Tensor

# Speeding up generation by caching keys and values


In [16]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# Tokenize the input prompt
encoded = tokenizer(prompt, return_tensors="pt")
input_ids=encoded['input_ids'].to(device)
attention_mask=encoded['attention_mask'].to(device)


In [17]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  attention_mask=attention_mask,
  max_new_tokens=100,
  use_cache=True
)

10 s ± 1.93 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  attention_mask=attention_mask,
  max_new_tokens=100,
  use_cache=False
)

33.1 s ± 4.99 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
